<a href="https://colab.research.google.com/github/yl4579/StyleTTS2/blob/main/Colab/StyleTTS2_SecondStage_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# StyleTTS 2 Stage-Two Inference

This notebook mirrors the loading and preprocessing pipeline from `train_second.py` to run stage-two StyleTTS 2 inference. It lets you choose a configuration file, automatically restore the most recent checkpoint from a directory (or load a specific `.pth` file), change the eSpeak phonemizer language, and control diffusion sampling parameters while auditioning the generated audio directly in the browser.


In [ ]:

%%capture
!pip install --quiet SoundFile torchaudio munch torch pydub pyyaml librosa nltk matplotlib accelerate transformers phonemizer einops einops-exts tqdm typing-extensions noisereduce git+https://github.com/resemble-ai/monotonic_align.git
!sudo apt-get update
!sudo apt-get install -y espeak-ng



### Repository setup

Clone the repository on Colab (if needed) and add it to `sys.path` so the modules imported in `train_second.py` can be reused for inference.


In [ ]:

import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/yl4579/StyleTTS2.git"
ROOT = pathlib.Path.cwd()

if (ROOT / "train_second.py").exists() and (ROOT / "Configs").exists():
    repo_path = ROOT
elif (ROOT / "StyleTTS2").is_dir():
    repo_path = ROOT / "StyleTTS2"
else:
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_path = ROOT / "StyleTTS2"

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.append(str(repo_path))

print(f"Working directory: {os.getcwd()}")



### Configure paths and phonemizer language

Provide the configuration YAML used for stage-two training and point `model_path` either to a specific checkpoint file or to the directory that contains `epoch_2nd_XXXXX.pth` files. When a directory is given the most recent checkpoint is selected automatically. The phonemizer language can be changed to any eSpeak identifier (e.g. `en-us`, `en-gb`, `fr-fr`, `de`, `es`, `it`, ...).


In [ ]:

# @title Inference configuration { display-mode: "form" }
config_path = "Configs/config.yml"  # @param {type:"string"}
model_path = "Models/LJSpeech"  # @param {type:"string"}
phonemizer_language = "en-us"  # @param {type:"string"}


In [ ]:

import glob
import math
import re
import time
import warnings
from pathlib import Path

import librosa
import nltk
import numpy as np
import phonemizer
import soundfile as sf
import torch
import torchaudio
import yaml
from IPython.display import Audio, display
from munch import Munch

from Utils.PLBERT.util import load_plbert
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule
from models import build_model, load_ASR_models, load_F0_models, load_checkpoint
from phoneme_dictionary import (
    DEFAULT_DICTIONARY_PATH,
    infer_phoneme_dictionary_token_count,
    resolve_phoneme_dictionary_settings,
)
from text_utils import TextCleaner
from utils import length_to_mask, recursive_munch

nltk.download("punkt", quiet=True)

def _resolve_path(path_value: str, base_dir: Path) -> Path:
    candidate = Path(path_value).expanduser()
    if not candidate.is_absolute():
        candidate = (base_dir / candidate).resolve()
    return candidate

def _resolve_checkpoint_path(path_value: str, base_dir: Path) -> Path:
    target = _resolve_path(path_value, base_dir)
    if target.is_file():
        return target
    if target.is_dir():
        candidates = sorted(target.glob("*.pth"))
        if not candidates:
            raise FileNotFoundError(f"No .pth checkpoints found in {target}")

        def _score(path: Path):
            match = re.search(r"(\d+)(?=\.pth$)", path.name)
            epoch = int(match.group(1)) if match else -1
            return (epoch, path.stat().st_mtime)

        best = max(candidates, key=_score)
        print(f"Selected checkpoint: {best}")
        return best
    raise FileNotFoundError(f"Checkpoint path {target} does not exist")

repo_root = Path.cwd()
config_file = _resolve_path(config_path, repo_root)
checkpoint_file = _resolve_checkpoint_path(model_path, repo_root)

with open(config_file, "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

spect_params = config.get("preprocess_params", {}).get("spect_params", {})
sample_rate = config.get("preprocess_params", {}).get("sr", 24000)

data_params = config.get("data_params") or {}
asr_config_path = config.get("ASR_config") or None

dictionary_source, dictionary_settings = resolve_phoneme_dictionary_settings(
    data_params=data_params,
    asr_config_path=_resolve_path(asr_config_path, repo_root) if asr_config_path else None,
)

dictionary_source = dictionary_source or DEFAULT_DICTIONARY_PATH
dictionary_token_count = infer_phoneme_dictionary_token_count(
    dictionary_source,
    dictionary_settings,
)

textcleaner = TextCleaner(dictionary_source, dictionary_config=dictionary_settings)

print(f"Loading auxiliary models from {repo_root} ...")
text_aligner = load_ASR_models(
    config.get("ASR_path"),
    asr_config_path,
    dictionary_path=dictionary_source,
    dictionary_config=dictionary_settings,
)
pitch_extractor = load_F0_models(config.get("F0_path"), config.get("F0_config"))
plbert = load_plbert(config.get("PLBERT_dir"))

model_params = recursive_munch(config["model_params"])
if dictionary_token_count is not None:
    configured_tokens = getattr(model_params, "n_token", None)
    if not isinstance(configured_tokens, int) or configured_tokens < dictionary_token_count:
        print(
            f"Adjusting n_token from {configured_tokens} to {dictionary_token_count} to match the phoneme dictionary",
        )
        model_params.n_token = dictionary_token_count

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = build_model(model_params, text_aligner, pitch_extractor, plbert)
model, _, _, _ = load_checkpoint(model, None, str(checkpoint_file))

for key in model:
    module = model[key]
    module.eval()
    model[key] = module.to(device)

diffusion_module = model["diffusion"]
diffusion_impl = getattr(diffusion_module, "diffusion", diffusion_module)
sampler = DiffusionSampler(
    diffusion_impl,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),
    clamp=False,
)

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=spect_params.get("n_fft", 2048),
    win_length=spect_params.get("win_length", 1200),
    hop_length=spect_params.get("hop_length", 300),
    n_mels=model_params.n_mels,
)
_mel_mean, _mel_std = -4.0, 4.0

def waveform_to_mel(waveform: np.ndarray) -> torch.Tensor:
    wave_tensor = torch.from_numpy(waveform).float()
    mel_tensor = mel_transform(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - _mel_mean) / _mel_std
    return mel_tensor

def update_phonemizer(language_code: str):
    global global_phonemizer
    if not language_code:
        language_code = config.get("inference_params", {}).get("phonemizer_language", "en-us")
    try:
        global_phonemizer = phonemizer.backend.EspeakBackend(
            language=language_code,
            preserve_punctuation=True,
            with_stress=True,
            words_mismatch="ignore",
        )
    except Exception as exc:
        warnings.warn(f"Failed to initialise eSpeak language '{language_code}': {exc}")
        fallback = config.get("inference_params", {}).get("phonemizer_language", "en-us")
        global_phonemizer = phonemizer.backend.EspeakBackend(
            language=fallback,
            preserve_punctuation=True,
            with_stress=True,
            words_mismatch="ignore",
        )
    else:
        print(f"Phonemizer language set to: {language_code}")
    return global_phonemizer

global_phonemizer = update_phonemizer(phonemizer_language)

style_dim = model_params.style_dim

print("Models ready. Diffusion sampler initialised.")



### Utility functions

Helper routines to convert text into phoneme tokens, build alignments, compute reference style embeddings, and run the diffusion decoder.


In [ ]:

from nltk import word_tokenize

def phonemize_text(text: str, backend=None) -> str:
    backend = backend or global_phonemizer
    cleaned = text.strip().replace('"', "")
    phonemes = backend.phonemize([cleaned])
    tokens = word_tokenize(phonemes[0])
    return " ".join(tokens)

def build_token_tensor(text: str) -> torch.LongTensor:
    phoneme_sequence = phonemize_text(text)
    token_ids = textcleaner(phoneme_sequence)
    token_ids.insert(0, textcleaner.word_index_dictionary.get("$", 0))
    return torch.LongTensor(token_ids).unsqueeze(0).to(device)

def build_alignment(predicted_durations: torch.Tensor, total_tokens: int) -> torch.Tensor:
    length = int(predicted_durations.sum().item())
    alignment = torch.zeros(total_tokens, length, device=device)
    cursor = 0
    for index in range(total_tokens):
        width = int(predicted_durations[index].item())
        alignment[index, cursor:cursor + width] = 1.0
        cursor += width
    return alignment

def compute_reference_style(audio_path: str, trim_db: float = 30.0) -> torch.Tensor:
    waveform, sr_in = librosa.load(audio_path, sr=sample_rate)
    if trim_db is not None:
        waveform, _ = librosa.effects.trim(waveform, top_db=trim_db)
    mel_tensor = waveform_to_mel(waveform).to(device)
    with torch.no_grad():
        style = model["style_encoder"](mel_tensor.unsqueeze(1))
        prosody = model["predictor_encoder"](mel_tensor.unsqueeze(1))
    return torch.cat([style, prosody], dim=1)

def synthesize(
    text: str,
    diffusion_steps: int = 30,
    embedding_scale: float = 1.0,
    noise_seed: int | None = None,
    reference_style: torch.Tensor | None = None,
    alpha: float = 0.3,
    beta: float = 0.7,
    phonemizer_backend=None,
) -> np.ndarray:
    tokens = build_token_tensor(text)
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model["text_encoder"](tokens, input_lengths, text_mask)
        bert_dur = model["bert"](tokens, attention_mask=(~text_mask).int())
        d_en = model["bert_encoder"](bert_dur).transpose(-1, -2)

        generator = None
        if noise_seed is not None:
            generator = torch.Generator(device=device)
            generator.manual_seed(int(noise_seed))

        noise = torch.randn((1, style_dim * 2), generator=generator, device=device)
        sampler_kwargs = dict(
            embedding=bert_dur,
            embedding_scale=float(embedding_scale),
            num_steps=int(diffusion_steps),
        )
        if reference_style is not None:
            reference_style = reference_style.to(device)
            sampler_kwargs["features"] = reference_style
        elif model_params.multispeaker:
            raise ValueError("This checkpoint expects a reference style embedding. Provide `reference_style`.")

        s_pred = sampler(noise=noise, **sampler_kwargs).squeeze(0)
        ref = s_pred[:, :style_dim]
        sty = s_pred[:, style_dim:]

        if reference_style is not None:
            ref = float(alpha) * ref + (1.0 - float(alpha)) * reference_style[:, :style_dim]
            sty = float(beta) * sty + (1.0 - float(beta)) * reference_style[:, style_dim:]

        d = model["predictor"].text_encoder(d_en, sty, input_lengths, text_mask)
        x, _ = model["predictor"].lstm(d)
        duration = model["predictor"].duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        text_stripped = text.strip()
        if not text_stripped or not text_stripped[-1].isalnum():
            pred_dur[-1] = 1

        alignment = build_alignment(pred_dur, input_lengths.item())

        en = d.transpose(-1, -2) @ alignment.unsqueeze(0)
        if model_params.decoder.type == "hifigan":
            en = torch.cat([en[:, :, :1], en[:, :, :-1]], dim=2)

        F0_pred, N_pred = model["predictor"].F0Ntrain(en, sty)

        asr = t_en @ alignment.unsqueeze(0)
        if model_params.decoder.type == "hifigan":
            asr = torch.cat([asr[:, :, :1], asr[:, :, :-1]], dim=2)

        out = model["decoder"](asr, F0_pred, N_pred, ref)
    return out.squeeze().cpu().numpy()



### Run inference

Fill in the text prompt and (optionally) a reference audio file. The cell plays the generated waveform, reports the real-time factor (RTF), and saves the audio to disk when a filename is supplied.


In [ ]:

# @title Generate speech { display-mode: "form" }
prompt = "StyleTTS 2 generates speech with rich prosody."  # @param {type:"string"}
diffusion_steps = 30  # @param {type:"slider", min:1, max:200, step:1}
embedding_scale = 1.0  # @param {type:"number"}
noise_seed = -1  # @param {type:"integer"}
reference_audio_path = ""  # @param {type:"string"}
alpha = 0.3  # @param {type:"number"}
beta = 0.7  # @param {type:"number"}
output_wav_path = ""  # @param {type:"string"}

reference_embedding = None
if reference_audio_path:
    reference_embedding = compute_reference_style(reference_audio_path)

seed_value = None if noise_seed < 0 else int(noise_seed)

start_time = time.time()
audio_samples = synthesize(
    prompt,
    diffusion_steps=diffusion_steps,
    embedding_scale=embedding_scale,
    noise_seed=seed_value,
    reference_style=reference_embedding,
    alpha=alpha,
    beta=beta,
)
elapsed = time.time() - start_time
rtf = elapsed / (len(audio_samples) / sample_rate)
print(f"RTF: {rtf:.4f}")

if output_wav_path:
    sf.write(output_wav_path, audio_samples, sample_rate)
    print(f"Saved waveform to: {output_wav_path}")

display(Audio(audio_samples, rate=sample_rate, normalize=False))
